# Preparación y estructuración de datos FiveM

Este flujo trabaja con las cinco tablas relacionales ampliadas de `data/examples/fivem/expanded/`. Son **datos sintéticos reproducibles**, no telemetría de un servidor real. Los CSV originales se conservan sin cambios; las versiones normalizadas y las tablas de análisis se escriben en `data/processed/`.

Pregunta analítica: ¿qué actividades, trabajos y zonas concentran más sesiones, y qué relación descriptiva se observa entre los movimientos económicos y la actividad de juego? Las asociaciones calculadas no implican causalidad ni describen una comunidad real.

## 1. Entorno y fuentes

El análisis principal enlaza perfiles, sesiones y transacciones por `player_id`. Vehículos y rendimiento se limpian y exportan por separado porque no son necesarios para responder esta pregunta. No se combinan fuentes de poblaciones distintas.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

PROJECT_ROOT = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "data" / "examples" / "fivem" / "expanded").is_dir()),
    None,
)
assert PROJECT_ROOT is not None, "Abre el Notebook desde la carpeta del repositorio."
SOURCE_DIR = PROJECT_ROOT / "data" / "examples" / "fivem" / "expanded"
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

SOURCE_FILES = {
    "players": "players.csv",
    "sessions": "sessions.csv",
    "transactions": "transactions.csv",
    "vehicles": "vehicles.csv",
    "server_performance": "server_performance.csv",
}

missing_files = [name for name in SOURCE_FILES.values() if not (SOURCE_DIR / name).is_file()]
assert not missing_files, f"Faltan archivos de origen: {missing_files}. Regenera los CSV con scripts/generar_datasets_ampliados.py."
raw = {name: pd.read_csv(SOURCE_DIR / filename) for name, filename in SOURCE_FILES.items()}
print(f"Pandas {pd.__version__} | Origen: {SOURCE_DIR.relative_to(PROJECT_ROOT)} | Salida: {OUTPUT_DIR.relative_to(PROJECT_ROOT)}")

Pandas 3.1.1 | Origen: data\examples\fivem\expanded | Salida: data\processed


## 2. Estado inicial de calidad

Se cuantifican dimensiones, celdas vacías y duplicados exactos antes de transformar. Los duplicados por clave se revisan aparte: una clave repetida no se elimina automáticamente si sus filas tienen información diferente.

In [2]:
def quality_snapshot(frames: dict[str, pd.DataFrame], stage: str) -> pd.DataFrame:
    records = []
    for name, frame in frames.items():
        records.append({
            "dataset": name,
            "stage": stage,
            "rows": len(frame),
            "columns": len(frame.columns),
            "missing_cells": int(frame.isna().sum().sum()),
            "exact_duplicate_rows": int(frame.duplicated().sum()),
        })
    return pd.DataFrame(records)

quality_before = quality_snapshot(raw, "before")
display(quality_before)
display(pd.DataFrame({name: frame.dtypes.astype(str) for name, frame in raw.items()}))

           dataset  stage  rows  columns  missing_cells  exact_duplicate_rows
           players before  1515       11            156                    15
          sessions before  8000       10            270                     0
      transactions before 12000        7            279                     0
          vehicles before  2200        9            493                     0
server_performance before  2840        7            138                     0
players sessions transactions vehicles server_performance
    NaN      NaN          str      NaN                NaN
  int64      NaN          NaN      NaN                NaN
    NaN      NaN          str      NaN                NaN
float64      NaN          NaN      NaN                NaN
  int64      NaN          NaN      NaN                NaN
float64      NaN          NaN      NaN                NaN
    NaN      str          NaN      NaN                NaN
    NaN      NaN          NaN      NaN            float64
    NaN   

## 3. Limpieza y estandarización

Las funciones reutilizables se encadenan con `pipe()`. Se quitan solo filas completamente duplicadas. Las categorías se recortan y convierten a minúsculas; fechas y números se parsean sin imputar. Valores fuera de límites explícitos se dejan como faltantes para análisis posteriores, sin borrar silenciosamente las filas fuente. Los límites (duración de sesión 1–720 minutos, FPS 1–60 y CPU 0–100%) son filtros exploratorios de esta simulación, no reglas universales de un servidor.

In [3]:
def remove_exact_duplicates(frame: pd.DataFrame) -> pd.DataFrame:
    return frame.drop_duplicates().copy()


def normalize_category(values: pd.Series) -> pd.Series:
    normalized = values.astype("string").str.strip().str.lower()
    return normalized.mask(normalized.eq(""), pd.NA)


def clean_players(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame.pipe(remove_exact_duplicates)
        .assign(
            player_id=lambda data: pd.to_numeric(data["player_id"], errors="raise").astype("Int64"),
            job=lambda data: normalize_category(data["job"]),
            cash=lambda data: pd.to_numeric(data["cash"], errors="coerce"),
            bank=lambda data: pd.to_numeric(data["bank"], errors="coerce"),
            age_character=lambda data: pd.to_numeric(data["age_character"], errors="coerce").where(
                lambda age: age.between(18, 100)
            ),
            registered_at=lambda data: pd.to_datetime(data["registered_at"], errors="coerce"),
            last_seen=lambda data: pd.to_datetime(data["last_seen"], errors="coerce"),
            banned=lambda data: pd.to_numeric(data["banned"], errors="coerce").map({0: False, 1: True}),
        )
    )


def clean_sessions(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame.pipe(remove_exact_duplicates)
        .assign(
            session_id=lambda data: pd.to_numeric(data["session_id"], errors="raise").astype("Int64"),
            player_id=lambda data: pd.to_numeric(data["player_id"], errors="coerce").astype("Int64"),
            connect_time=lambda data: pd.to_datetime(data["connect_time"], errors="coerce"),
            disconnect_time=lambda data: pd.to_datetime(data["disconnect_time"], errors="coerce"),
            duration_min=lambda data: pd.to_numeric(data["duration_min"], errors="coerce"),
            main_activity=lambda data: normalize_category(data["main_activity"]),
            zone=lambda data: data["zone"].astype("string").str.strip().replace("", pd.NA),
            pos_x=lambda data: pd.to_numeric(data["pos_x"], errors="coerce"),
            pos_y=lambda data: pd.to_numeric(data["pos_y"], errors="coerce"),
            pos_z=lambda data: pd.to_numeric(data["pos_z"], errors="coerce"),
        )
    )


def clean_transactions(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame.pipe(remove_exact_duplicates)
        .assign(
            transaction_id=lambda data: pd.to_numeric(data["transaction_id"], errors="raise").astype("Int64"),
            player_id=lambda data: pd.to_numeric(data["player_id"], errors="coerce").astype("Int64"),
            timestamp=lambda data: pd.to_datetime(data["timestamp"], errors="coerce"),
            type=lambda data: normalize_category(data["type"]),
            amount=lambda data: pd.to_numeric(
                data["amount"].astype("string").str.replace("$", "", regex=False).str.replace(",", "", regex=False),
                errors="coerce",
            ),
            account=lambda data: normalize_category(data["account"]),
            source_job=lambda data: normalize_category(data["source_job"]),
        )
    )


def clean_vehicles(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame.pipe(remove_exact_duplicates)
        .assign(
            vehicle_id=lambda data: pd.to_numeric(data["vehicle_id"], errors="raise").astype("Int64"),
            owner_player_id=lambda data: pd.to_numeric(data["owner_player_id"], errors="coerce").astype("Int64"),
            model=lambda data: normalize_category(data["model"]),
            plate=lambda data: data["plate"].astype("string").str.strip().str.upper(),
            price_paid=lambda data: pd.to_numeric(data["price_paid"], errors="coerce").where(
                lambda price: price.ge(0)
            ),
            garage=lambda data: normalize_category(data["garage"]),
            fuel=lambda data: pd.to_numeric(data["fuel"], errors="coerce").where(
                lambda fuel: fuel.between(0, 100)
            ),
            engine_health=lambda data: pd.to_numeric(data["engine_health"], errors="coerce").where(
                lambda health: health.between(0, 1000)
            ),
            purchase_date=lambda data: pd.to_datetime(data["purchase_date"], errors="coerce"),
        )
    )


def clean_server_performance(frame: pd.DataFrame) -> pd.DataFrame:
    return (
        frame.pipe(remove_exact_duplicates)
        .assign(
            timestamp=lambda data: pd.to_datetime(data["timestamp"], errors="coerce"),
            players_online=lambda data: pd.to_numeric(data["players_online"], errors="coerce").where(
                lambda value: value.ge(0)
            ),
            server_fps=lambda data: pd.to_numeric(data["server_fps"], errors="coerce").where(
                lambda value: value.between(1, 60)
            ),
            tickrate=lambda data: pd.to_numeric(data["tickrate"], errors="coerce").where(
                lambda value: value.between(1, 40)
            ),
            cpu_pct=lambda data: pd.to_numeric(data["cpu_pct"], errors="coerce").where(
                lambda value: value.between(0, 100)
            ),
            ram_mb=lambda data: pd.to_numeric(data["ram_mb"], errors="coerce").where(
                lambda value: value.gt(0)
            ),
            heaviest_resource=lambda data: normalize_category(data["heaviest_resource"]),
        )
    )


cleaned = {
    "players": raw["players"].pipe(clean_players),
    "sessions": raw["sessions"].pipe(clean_sessions),
    "transactions": raw["transactions"].pipe(clean_transactions),
    "vehicles": raw["vehicles"].pipe(clean_vehicles),
    "server_performance": raw["server_performance"].pipe(clean_server_performance),
}

quality_after = quality_snapshot(cleaned, "after")
quality_comparison = pd.concat([quality_before, quality_after], ignore_index=True)
display(quality_comparison)

           dataset  stage  rows  columns  missing_cells  exact_duplicate_rows
           players before  1515       11            156                    15
          sessions before  8000       10            270                     0
      transactions before 12000        7            279                     0
          vehicles before  2200        9            493                     0
server_performance before  2840        7            138                     0
           players  after  1500       11            160                     0
          sessions  after  8000       10            280                     0
      transactions  after 12000        7            309                     0
          vehicles  after  2200        9            753                     0
server_performance  after  2840        7            182                     0


## 4. Validaciones y evidencia de limpieza

Los registros inválidos para un cálculo específico permanecen en las tablas limpias con valores nulos en los campos problemáticos. Se excluyen únicamente de ese cálculo. El pipeline conserva las columnas de entrada para poder comparar calidad; al exportar perfiles elimina los identificadores simulados (`license`, `discord_id`, `username`). `player_id` se conserva como clave ficticia para las relaciones.

In [4]:
players_clean = cleaned["players"]
sessions_clean = cleaned["sessions"]
transactions_clean = cleaned["transactions"]
vehicles_clean = cleaned["vehicles"]
performance_clean = cleaned["server_performance"]

assert players_clean["player_id"].notna().all()
assert players_clean["player_id"].is_unique, "Claves de jugador repetidas después de quitar duplicados exactos."
assert sessions_clean["session_id"].notna().all() and sessions_clean["session_id"].is_unique
assert transactions_clean["transaction_id"].notna().all() and transactions_clean["transaction_id"].is_unique
assert vehicles_clean["vehicle_id"].notna().all() and vehicles_clean["vehicle_id"].is_unique
assert not any(frame.duplicated().any() for frame in cleaned.values())

known_player_ids = set(players_clean["player_id"].dropna().astype(int))
duration_valid = sessions_clean["duration_min"].between(1, 720)
session_dates_valid = sessions_clean["connect_time"].notna()
session_players_valid = sessions_clean["player_id"].isin(known_player_ids)
valid_sessions = sessions_clean.loc[duration_valid & session_dates_valid & session_players_valid].copy()
orphan_sessions = sessions_clean.loc[~session_players_valid]

# Extreme amounts were planted as quality cases in this synthetic generator.
# The threshold is an analysis exclusion, not a financial business rule.
valid_amount = transactions_clean["amount"].notna() & transactions_clean["amount"].abs().lt(100_000_000)
valid_transactions = transactions_clean.loc[
    valid_amount
    & transactions_clean["timestamp"].notna()
    & transactions_clean["player_id"].isin(known_player_ids)
].copy()

assert valid_sessions["duration_min"].between(1, 720).all()
assert valid_sessions["player_id"].isin(known_player_ids).all()
assert valid_transactions["amount"].abs().lt(100_000_000).all()
assert valid_transactions["player_id"].isin(known_player_ids).all()

cleaning_evidence = pd.DataFrame([
    {"dataset": name, "source_rows": len(raw[name]), "exact_duplicates_removed": int(raw[name].duplicated().sum()),
     "processed_rows": len(cleaned[name]), "missing_cells_before": int(raw[name].isna().sum().sum()),
     "missing_cells_after": int(cleaned[name].isna().sum().sum())}
    for name in raw
])
display(cleaning_evidence)
print(f"Sesiones elegibles para métricas: {len(valid_sessions):,} | referencias huérfanas en sesiones: {len(orphan_sessions):,}")
print(f"Transacciones elegibles para métricas económicas: {len(valid_transactions):,} de {len(transactions_clean):,}")

           dataset  source_rows  exact_duplicates_removed  processed_rows  missing_cells_before  missing_cells_after
           players         1515                        15            1500                   156                  160
          sessions         8000                         0            8000                   270                  280
      transactions        12000                         0           12000                   279                  309
          vehicles         2200                         0            2200                   493                  753
server_performance         2840                         0            2840                   138                  182
Sesiones elegibles para métricas: 7,905 | referencias huérfanas en sesiones: 30
Transacciones elegibles para métricas económicas: 11,741 de 12,000


## 5. Filtrado, agrupación e indicadores

El análisis utiliza sesiones con duración entre 1 y 720 minutos, fecha interpretable y jugador existente. Las zonas faltantes se agrupan como `unknown`, no se imputan. Se presentan conteo de sesiones, jugadores distintos y duración promedio por actividad, trabajo y zona.

In [5]:
def summarize_sessions(frame: pd.DataFrame, dimension: str) -> pd.DataFrame:
    prepared = frame.assign(**{dimension: frame[dimension].fillna("unknown")})
    return (
        prepared.groupby(dimension, as_index=False, dropna=False)
        .agg(
            session_count=("session_id", "size"),
            player_count=("player_id", "nunique"),
            total_playtime_minutes=("duration_min", "sum"),
            average_session_minutes=("duration_min", "mean"),
        )
        .sort_values(["session_count", dimension], ascending=[False, True])
        .reset_index(drop=True)
    )


sessions_by_activity = summarize_sessions(valid_sessions, "main_activity")
sessions_with_job = valid_sessions.merge(
    players_clean[["player_id", "job"]], on="player_id", how="left", validate="many_to_one"
)
assert len(sessions_with_job) == len(valid_sessions)
sessions_by_job = summarize_sessions(sessions_with_job, "job")
sessions_by_zone = summarize_sessions(valid_sessions, "zone")

display(sessions_by_activity.head(10))
display(sessions_by_job.head(10))
display(sessions_by_zone.head(10))

indicator_summary = pd.DataFrame([
    {"indicator": "Sesiones válidas", "value": len(valid_sessions), "unit": "sesiones"},
    {"indicator": "Jugadores con sesión válida", "value": valid_sessions["player_id"].nunique(), "unit": "jugadores"},
    {"indicator": "Duración media por sesión", "value": valid_sessions["duration_min"].mean(), "unit": "minutos"},
    {"indicator": "Zonas sin clasificar", "value": int(valid_sessions["zone"].isna().sum()), "unit": "sesiones"},
])
display(indicator_summary)

main_activity  session_count  player_count  total_playtime_minutes  average_session_minutes
police_patrol           1024           751                152110.4               148.545312
 drug_dealing           1012           751                161763.3               159.845158
       mining           1009           732                151541.4               150.189693
      fishing            984           696                145315.1               147.677947
     mechanic            982           735                148150.3               150.865886
       racing            976           714                150889.5               154.599898
         taxi            971           701                149419.4               153.881977
         none            947           713                143685.3               151.726822
        job  session_count  player_count  total_playtime_minutes  average_session_minutes
 unemployed           1686           316                264009.9               156

## 6. Consolidación relacional y proxy de recurrencia

Se agregan sesiones y movimientos económicos a una fila por `player_id` y se combinan con perfiles mediante uniones `one_to_one`. La recurrencia a 7 días es una **proxy**: una segunda sesión al menos siete días después de la primera, solo entre jugadores con siete días observables desde su primera sesión. No es una métrica de retención de producto validada. En transacciones se conserva el signo original; positivo se interpreta como ingreso y negativo como gasto, según la convención del generador.

In [6]:
session_metrics = (
    valid_sessions.assign(session_day=valid_sessions["connect_time"].dt.date)
    .groupby("player_id", as_index=False)
    .agg(
        session_count=("session_id", "nunique"),
        total_playtime_minutes=("duration_min", "sum"),
        average_session_minutes=("duration_min", "mean"),
        active_days=("session_day", "nunique"),
        first_session=("connect_time", "min"),
        last_session=("connect_time", "max"),
    )
)
observation_end = valid_sessions["connect_time"].max()
session_metrics["eligible_7d_observation"] = (
    session_metrics["first_session"] <= observation_end - pd.Timedelta(days=7)
)
session_metrics["returned_after_7d"] = (
    session_metrics["last_session"] >= session_metrics["first_session"] + pd.Timedelta(days=7)
)
session_metrics["repeat_7d_proxy"] = (
    session_metrics["eligible_7d_observation"] & session_metrics["returned_after_7d"]
)

transaction_metrics = (
    valid_transactions.assign(
        income=lambda data: data["amount"].clip(lower=0),
        expense=lambda data: -data["amount"].clip(upper=0),
    )
    .groupby("player_id", as_index=False)
    .agg(
        transaction_count=("transaction_id", "nunique"),
        signed_net_transactions=("amount", "sum"),
        gross_income=("income", "sum"),
        gross_expenses=("expense", "sum"),
    )
)

player_analysis = (
    players_clean[["player_id", "job", "cash", "bank", "age_character", "registered_at", "last_seen", "banned"]]
    .merge(session_metrics, on="player_id", how="left", validate="one_to_one")
    .merge(transaction_metrics, on="player_id", how="left", validate="one_to_one")
)
for column in ["session_count", "total_playtime_minutes", "active_days", "transaction_count",
               "signed_net_transactions", "gross_income", "gross_expenses"]:
    player_analysis[column] = player_analysis[column].fillna(0)
player_analysis["eligible_7d_observation"] = player_analysis["eligible_7d_observation"].fillna(False)
player_analysis["repeat_7d_proxy"] = player_analysis["repeat_7d_proxy"].fillna(False)
player_analysis["average_minutes_per_active_day"] = (
    player_analysis["total_playtime_minutes"]
    .div(player_analysis["active_days"].replace(0, np.nan))
)

assert player_analysis["player_id"].is_unique
assert len(player_analysis) == len(players_clean), "La consolidación debe conservar todos los perfiles."
assert set(player_analysis["player_id"].astype(int)) == known_player_ids
assert player_analysis["signed_net_transactions"].notna().all()

eligible_players = player_analysis.loc[player_analysis["eligible_7d_observation"]]
repeat_rate_7d = eligible_players["repeat_7d_proxy"].mean() if len(eligible_players) else np.nan
economic_activity_correlation = player_analysis[[
    "signed_net_transactions", "total_playtime_minutes"
]].corr(method="spearman").iloc[0, 1]

print(f"Perfiles consolidados: {len(player_analysis):,} | sesiones válidas unidas: {len(valid_sessions):,}")
print(f"Jugadores elegibles para proxy a 7 días: {len(eligible_players):,} | proxy de repetición: {repeat_rate_7d:.1%}")
print(f"Correlación Spearman (neto de transacciones y minutos): {economic_activity_correlation:.3f}")
display(player_analysis.head())

Perfiles consolidados: 1,500 | sesiones válidas unidas: 7,905
Jugadores elegibles para proxy a 7 días: 1,488 | proxy de repetición: 96.5%
Correlación Spearman (neto de transacciones y minutos): -0.004
 player_id         job    cash    bank  age_character registered_at  last_seen  banned  session_count  total_playtime_minutes  average_session_minutes  active_days       first_session        last_session eligible_7d_observation returned_after_7d repeat_7d_proxy  transaction_count  signed_net_transactions  gross_income  gross_expenses  average_minutes_per_active_day
         1     trucker   983.0 22518.0           58.0    2026-06-13 2026-07-21   False            4.0                   449.5               112.375000          4.0 2026-07-30 22:53:46 2026-10-26 05:20:24                    True              True            True               11.0                  12811.0       29593.0         16782.0                      112.375000
         2 gang_member  4764.0 31869.0           61.0    2026-0

## 7. Exportación, integridad e interpretación

Las tablas se exportan como CSV para continuar el análisis. Los faltantes de dimensiones se mantienen; las tablas de análisis excluyen explícitamente registros no válidos para cada indicador. La tabla de vehículos no se une a la tabla de jugadores porque su relación de propietarios no aporta variables necesarias a la pregunta. El rendimiento se conserva como serie temporal independiente: no existe una clave de evento que permita asignarlo a sesiones de forma fiable.

In [7]:
players_export = players_clean.drop(columns=["license", "discord_id", "username"], errors="ignore")
assert not any(column in players_export.columns for column in ["license", "discord_id", "username"])

outputs = {
    "players_clean.csv": players_export,
    "sessions_clean.csv": sessions_clean,
    "transactions_clean.csv": transactions_clean,
    "vehicles_clean.csv": vehicles_clean,
    "server_performance_clean.csv": performance_clean,
    "sessions_by_activity.csv": sessions_by_activity,
    "sessions_by_job.csv": sessions_by_job,
    "sessions_by_zone.csv": sessions_by_zone,
    "player_analysis.csv": player_analysis,
    "quality_comparison.csv": quality_comparison,
}

for filename, frame in outputs.items():
    frame.to_csv(OUTPUT_DIR / filename, index=False)

assert len(pd.read_csv(OUTPUT_DIR / "players_clean.csv")) == len(players_clean)
assert len(pd.read_csv(OUTPUT_DIR / "player_analysis.csv")) == len(players_clean)
assert len(pd.read_csv(OUTPUT_DIR / "sessions_clean.csv")) == len(sessions_clean)
assert (OUTPUT_DIR / "player_analysis.csv").stat().st_size > 0
print("Archivos exportados:")
for filename, frame in outputs.items():
    print(f"- {filename}: {len(frame):,} filas")

top_activity = sessions_by_activity.iloc[0]
top_zone = sessions_by_zone.loc[sessions_by_zone["zone"] != "unknown"].iloc[0]
print(
    f"\nInterpretación: en estos registros sintéticos, la actividad con más sesiones es "
    f"'{top_activity['main_activity']}' ({int(top_activity['session_count']):,}) y la zona observada "
    f"con más sesiones es '{top_zone['zone']}' ({int(top_zone['session_count']):,}). "
    f"El coeficiente económico es descriptivo ({economic_activity_correlation:.3f}); no demuestra "
    "que los movimientos económicos causen cambios en el tiempo de juego."
)

Archivos exportados:
- players_clean.csv: 1,500 filas
- sessions_clean.csv: 8,000 filas
- transactions_clean.csv: 12,000 filas
- vehicles_clean.csv: 2,200 filas
- server_performance_clean.csv: 2,840 filas
- sessions_by_activity.csv: 8 filas
- sessions_by_job.csv: 11 filas
- sessions_by_zone.csv: 9 filas
- player_analysis.csv: 1,500 filas
- quality_comparison.csv: 10 filas

Interpretación: en estos registros sintéticos, la actividad con más sesiones es 'police_patrol' (1,024) y la zona observada con más sesiones es 'Legion Square' (1,002). El coeficiente económico es descriptivo (-0.004); no demuestra que los movimientos económicos causen cambios en el tiempo de juego.


## 8. Conclusiones y límites

Las agrupaciones responden qué valores aparecen con mayor frecuencia en el escenario creado y las uniones conservan una fila por perfil, descartando del resumen de sesiones las claves huérfanas. La medida de recurrencia se calcula solo para el grupo con siete días de seguimiento posible. El signo de las transacciones y los límites de calidad provienen de supuestos del generador.

Como los datos se generaron aleatoriamente, estos resultados no son evidencia sobre FiveM real y no permiten inferir impacto de la economía ni retención auténtica. Para una conclusión empírica se requieren exportaciones autorizadas, diccionario de datos, reglas verificadas de signos/unidades y una ventana de observación definida. La fuente de rendimiento y la tabla de vehículos quedan estructuradas por separado porque no existe una relación temporal o analítica suficiente para fusionarlas sin fabricar asociaciones.